# edge-grader: stage-3 fine-tuning on Kaggle (QLoRA)

Trains a QLoRA adapter for Gemma 4 E2B or E4B on the SciEntsBank train split with `edge-grader-finetune`, converts it to GGUF with llama.cpp's `convert_lora_to_gguf.py`, then grades SciEntsBank answers with the **unchanged lab GGUF plus `--lora`**, the same way the Jetson will serve it. Design: `docs/superpowers/specs/2026-10-21-finetuning-design.md`.

**`SMOKE = True` is the Day 1 risk check:** 20 training steps, validation on 20 answers, 10 test_ua answers graded. It answers three questions: does Gemma 4 train on a T4, does the adapter convert, and does `llama-server` load it. **`SMOKE = False` is the real run:** up to 3 epochs (validation picks one), then 100 test_ua answers as a sanity check (the board runs are the result).

**Before running** (notebook sidebar):
1. **Settings → Accelerator:** GPU T4 x1.
2. **Settings → Internet:** on.
3. **Add-ons → Secrets:** `GITHUB_TOKEN`, read-only *Contents* on `gnotnek/edge-grader` only, as for the baseline notebook. Never paste it into a cell.

**What to bring back:** `results-finetune.zip` (section 7) goes into the repo. The adapter `adapters/<model>-ft.gguf` (Output panel) does **not**: after a real run, upload it to the private Kaggle dataset `edge-grader-adapters`.

In [ ]:
MODEL = "e2b"   # or "e4b"
SMOKE = True    # True: the Day 1 check (minutes). False: the real training run (hours)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and install it with uv (with the `finetune` group)

First, so a torch that can't see Kaggle's GPU stops the notebook before the 10–15 min llama.cpp build.

In [ ]:
import base64
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

REPO = "/kaggle/working/edge-grader"
# The token reaches git through the environment as an HTTP header, as actions/checkout does: never
# in the command line (a failed clone's error, shown in the notebook output, would print it) and
# never in .git/config.
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = {**os.environ, "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
       "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
del token, auth
shutil.rmtree(REPO, ignore_errors=True)
clone = subprocess.run(["git", "clone", "-q", "https://github.com/gnotnek/edge-grader.git", REPO],
                       env=env, capture_output=True, text=True, check=False)
del env
if clone.returncode != 0:
    raise RuntimeError(f"git clone failed (check the GITHUB_TOKEN secret): {clone.stderr.strip()}")
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q uv
!cd /kaggle/working/edge-grader && uv sync -q --group finetune && uv run pytest -q 2>&1 | tail -1
!cd /kaggle/working/edge-grader && uv run --group finetune python -c "import torch; assert torch.cuda.is_available(), f'torch {torch.__version__} (CUDA {torch.version.cuda}) sees no GPU: check the driver'; print('torch', torch.__version__, 'CUDA', torch.version.cuda, torch.cuda.get_device_name(0))"

## 2. Build llama-server with CUDA (about 10–15 min, once per session)

Pinned to `a3a1c47`, the same build as every Kaggle baseline, so the adapter is graded on the build the baseline was.

In [ ]:
%%bash
set -e
nvidia-smi -L || { echo "No GPU in this session: Settings -> Accelerator -> GPU, then restart the session"; exit 1; }
cd /tmp
# Pinned, so every run uses the same llama.cpp. Unpinned, Kaggle got whatever was newest that day:
# the stage-1 baseline ran on 86ea01d, the RAG-en runs on a3a1c47 (docs/results/rag.md).
LLAMA_CPP_COMMIT=a3a1c4747fdc0dcad40b3946108b89375d9a7d0e
if [ "$(git -C llama.cpp rev-parse HEAD 2>/dev/null)" != "$LLAMA_CPP_COMMIT" ]; then
  rm -rf llama.cpp && mkdir llama.cpp && cd llama.cpp
  git init -q && git remote add origin https://github.com/ggml-org/llama.cpp
  git fetch -q --depth 1 origin "$LLAMA_CPP_COMMIT" && git checkout -q FETCH_HEAD
  cd /tmp
fi
export PATH=/usr/local/cuda/bin:$PATH
CUDA_ROOT=$(dirname "$(dirname "$(readlink -f "$(command -v nvcc)")")")
# CMake can miss libcuda on Kaggle ("links to CUDA::cuda_driver but the target was not found"):
# the driver library sits outside the toolkit (e.g. /usr/local/nvidia/lib64). Find it and pass it in.
LIBCUDA=$( { ls "$CUDA_ROOT"/lib64/stubs/libcuda.so "$CUDA_ROOT"/targets/x86_64-linux/lib/stubs/libcuda.so 2>/dev/null;
             ldconfig -p | awk '/libcuda\.so/ {print $NF}';
             find /usr /opt -name 'libcuda.so*' 2>/dev/null; } | head -1 )
echo "nvcc: $(command -v nvcc)  CUDA_ROOT: $CUDA_ROOT  libcuda: ${LIBCUDA:-NOT FOUND}"
[ -n "$LIBCUDA" ] || { echo "GPU is on but libcuda was not found; paste this output to debug"; exit 1; }
rm -rf llama.cpp/build   # a failed configure caches the bad result
# 75 = T4, 60 = P100. Building only these keeps compile time down.
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="60;75" -DLLAMA_CURL=OFF \
  -DCUDAToolkit_ROOT="$CUDA_ROOT" -DCMAKE_CUDA_COMPILER="$CUDA_ROOT/bin/nvcc" \
  -DCUDA_cuda_driver_LIBRARY="$LIBCUDA" > /tmp/cmake.log 2>&1 || { tail -30 /tmp/cmake.log; exit 1; }
cmake --build llama.cpp/build -j"$(nproc)" --target llama-server > /tmp/build.log 2>&1 || { tail -30 /tmp/build.log; exit 1; }
cd llama.cpp && echo "llama.cpp $(git rev-parse --short HEAD)" && ls -la build/bin/llama-server

## 3. Download the lab's model file

In [ ]:
from huggingface_hub import hf_hub_download

FILES = {
    "e2b": ("unsloth/gemma-4-E2B-it-qat-GGUF", "gemma-4-E2B-it-qat-UD-Q4_K_XL.gguf"),
    "e4b": ("unsloth/gemma-4-E4B-it-qat-GGUF", "gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf"),
}
GGUF = hf_hub_download(*FILES[MODEL], local_dir="/tmp/models")
GGUF

## 4. Chat template check

The training text has to be exactly what `llama-server` renders with `-rea off`. This asks the server's `/apply-template` for 5 training prompts and compares them with transformers' template (thinking off). **If it prints a mismatch, stop here:** training on a different template would grade something the server never sends.

In [ ]:
import os
import subprocess
import time
import urllib.error
import urllib.request

REPO = "/kaggle/working/edge-grader"
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     check=True, capture_output=True, text=True).stdout.split("\n")[0].split()[-1].lower()  # e.g. "t4"

def start_server(lora=None):
    # The baseline flags (thinking off), -ngl 99, and the adapter if given: nothing else changes.
    cmd = ["/tmp/llama.cpp/build/bin/llama-server", "-m", GGUF, "-c", "8192",
           "--host", "127.0.0.1", "--port", "8080", "-ngl", "99",
           "-rea", "off", "--reasoning-budget", "-1", "--cache-ram", "0",
           *(["--lora", lora] if lora else [])]
    log_path = f"/tmp/server-{MODEL}{'-ft' if lora else ''}.log"
    with open(log_path, "w") as log:
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(150):
        try:
            if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).status == 200:
                return proc, log_path
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            pass  # still loading
        if proc.poll() is not None:
            break
        time.sleep(2)
    proc.kill()
    raise RuntimeError(f"llama-server failed; see {log_path}")

CHECK = r'''  # raw: the \n escapes belong to the script, not to this cell
import json, sys, urllib.request
from transformers import AutoTokenizer
from edge_grader import data
from edge_grader.finetune import BASES, to_messages

base, revision = BASES[sys.argv[1]]
tok = AutoTokenizer.from_pretrained(base, revision=revision)
labels = data.labels_for("3way")
for item in data.load(split="train", scheme="3way")[:5]:
    msgs = to_messages(item, labels)[:1]
    req = urllib.request.Request("http://127.0.0.1:8080/apply-template", data=json.dumps({"messages": msgs}).encode(),
                                 headers={"Content-Type": "application/json"})
    server = json.load(urllib.request.urlopen(req))["prompt"].removeprefix(tok.bos_token or "")
    hf = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                 enable_thinking=False).removeprefix(tok.bos_token or "")
    if server != hf:
        print("MISMATCH\nserver:", repr(server), "\nhf:    ", repr(hf))
        sys.exit(1)
    full = tok.apply_chat_template(to_messages(item, labels), tokenize=False, enable_thinking=False)
    if not full.removeprefix(tok.bos_token or "").startswith(hf):
        print("the training conversation doesn't start with the prompt:", repr(full))
        sys.exit(1)
print("chat template: llama-server and transformers agree on 5 training prompts, and each training text extends its prompt")
'''
server, _ = start_server()
try:
    check = subprocess.run(["uv", "run", "--group", "finetune", "python", "-c", CHECK, MODEL], cwd=REPO,
                           capture_output=True, text=True, check=False)
    print(check.stdout, check.stderr[-2000:] if check.returncode else "")
finally:
    server.terminate()
    server.wait()
assert check.returncode == 0, "chat template mismatch: stop and look before training"

## 5. Train

Validation (10% of the train questions) picks the epoch. Watch the loss: `nan` or a loss that keeps growing means fp16 overflow on the T4 (note it, and stop).

In [ ]:
import shutil

ADAPTERS = f"/kaggle/working/adapters/{MODEL}"
shutil.rmtree(ADAPTERS, ignore_errors=True)  # a re-run must not find the last run's adapter
SMOKE_ARGS = "--max-steps 20 --val-n 20" if SMOKE else ""
!cd {REPO} && uv run --group finetune edge-grader-finetune --model {MODEL} --out {ADAPTERS} {SMOKE_ARGS}
assert os.path.exists(f"{ADAPTERS}/adapter/adapter_model.safetensors"), "training failed: see the output above"

## 6. Convert the adapter to GGUF, load it with `--lora`, and grade

The converter gets its own venv with llama.cpp's own requirements, and the base model's config files only (no weights) at the pinned revision.

In [ ]:
import sys

from huggingface_hub import snapshot_download

sys.path.insert(0, f"{REPO}/src")
from edge_grader.finetune import BASES

BASE_CONFIG = snapshot_download(BASES[MODEL][0], revision=BASES[MODEL][1],
                                allow_patterns=["*.json", "*.jinja"],
                                # no shard index: it would make the converter look for the weights
                                ignore_patterns=["*.index.json"], local_dir=f"/tmp/base-config-{MODEL}")
FT_GGUF = f"/kaggle/working/adapters/{MODEL}-ft.gguf"
!uv venv -q --clear /tmp/convert-venv && uv pip install -q --python /tmp/convert-venv/bin/python -r /tmp/llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
!/tmp/convert-venv/bin/python /tmp/llama.cpp/convert_lora_to_gguf.py {ADAPTERS}/adapter --base {BASE_CONFIG} --outfile {FT_GGUF} --outtype f16
assert os.path.exists(FT_GGUF), "conversion failed: see the output above (this decides main path vs fallback)"
!sha256sum {FT_GGUF}

In [ ]:
from pathlib import Path

N = 10 if SMOKE else 100
server, log_path = start_server(lora=FT_GGUF)
try:
    loaded = [line for line in Path(log_path).read_text().splitlines(keepends=True)
              if "lora" in line.lower()]
    print("".join(loaded[:5]) or "no LoRA line in the server log")
    !cd {REPO} && uv run edge-grader-baseline --label {MODEL}-kaggle-{GPU}-ft --split test_ua --n {N} --out results/kaggle
finally:
    server.terminate()
    server.wait()

In [ ]:
import json
from pathlib import Path

for f in sorted(Path(f"{REPO}/results/finetune").glob("*.json")):
    s = json.loads(f.read_text())
    print(f.name, "chosen epoch", s["chosen_epoch"])
    for h in s["history"]:
        print("  ", h)
for f in sorted(Path(f"{REPO}/results/kaggle").glob(f"baseline-{MODEL}-kaggle-*-ft-*.json")):
    r = json.loads(f.read_text())
    print(f.name, {k: r["metrics"][k] for k in ("n", "accuracy", "macro_f1", "unparsed", "errors")})

**Day 1 outcome (SMOKE = True), to report back:**
- the template check passed or printed a mismatch;
- training ran (loss finite) or failed on the T4 → stop and escalate;
- `convert_lora_to_gguf.py` wrote the GGUF, and the server log shows the adapter loading → main path; otherwise → fallback (merge and quantize, spec §5.2);
- 10 answers graded with `unparsed` 0. Accuracy after 20 steps means nothing yet.

## 7. Download the results

Zips `results/finetune/` and this run's `results/kaggle/` files. On your machine:
```bash
unzip -o ~/Downloads/results-finetune.zip -d results
git add results/finetune results/kaggle && git commit -m "Stage 3: <model> fine-tuning run on Kaggle"
```
The adapter GGUF is downloaded separately from the Output panel (`adapters/<model>-ft.gguf`), and not committed.

In [ ]:
import shutil
from pathlib import Path

from IPython.display import FileLink

STAGE = "/tmp/results-finetune"
shutil.rmtree(STAGE, ignore_errors=True)
shutil.copytree(f"{REPO}/results/finetune", f"{STAGE}/finetune")
shutil.copytree(f"{REPO}/results/kaggle", f"{STAGE}/kaggle",
                ignore=lambda d, names: [n for n in names if not (f"-{MODEL}-kaggle-" in n and "-ft-" in n) and n != "raw"
                                         and not Path(d, n).is_dir()])
shutil.make_archive("/kaggle/working/results-finetune", "zip", STAGE)
FileLink("results-finetune.zip")